## Transferir o notebook

In [1]:
%pip install -r requirements.txt

import kagglehub
path = kagglehub.dataset_download( "harlfoxem/housesalesprediction",                                                                   output_dir="data/kingcounty",)

print("Ficheiros do dataset em:", path)


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: /Users/anytng/Developer/Universidade/Masters/FundamentalsOfAI/FAI26_P02_G09/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/Users/anytng/Developer/Universidade/Masters/FundamentalsOfAI/FAI26_P02_G09/.venv/lib/python3.15/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ficheiros do dataset em: data/kingcounty


## Mudança de dataset

Originalmente usámos o Melbourne Housing Snapshot, mas trocámos para o King County House Sales porque em Melbourne metade das vendas não tinha área construída e não havia informação sobre o estado e a qualidade das casas. O King County não tem valores em falta e inclui o estado, a qualidade da construção e as renovações.

## EDA

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

df = pd.read_csv("data/kingcounty/kc_house_data.csv")
df["date"] = pd.to_datetime(df["date"].str[:8])  # as datas vêm no formato 20141013T000000

# Passamos as áreas de pés quadrados para metros quadrados, com nomes novos
AREAS = {"sqft_living": "AreaCasa", "sqft_lot": "AreaTerreno", "sqft_above": "AreaAcimaChao",
         "sqft_basement": "AreaCave", "sqft_living15": "AreaCasaVizinhos", "sqft_lot15": "AreaTerrenoVizinhos"}
df[list(AREAS)] = df[list(AREAS)] * 0.09290304
df = df.rename(columns=AREAS)

print(f"{df.shape[0]} linhas x {df.shape[1]} colunas")
print("Vendas entre", df["date"].min().date(), "e", df["date"].max().date())
df.head()

21613 linhas x 21 colunas
Vendas entre 2014-05-02 e 2015-05-27


,id,date,price,bedrooms,bathrooms,AreaCasa,AreaTerreno,floors,waterfront,view,condition,grade,AreaAcimaChao,AreaCave,yr_built,yr_renovated,zipcode,lat,long,AreaCasaVizinhos,AreaTerrenoVizinhos
0,7129300520,2014-10-13,221900.0,3,1.00,109.625587,524.902176,1.0,0,0,3,7,109.625587,0.000000,1955,0,98178,47.5112,-122.257,124.490074,524.902176
1,6414100192,2014-12-09,538000.0,3,2.25,238.760813,672.803816,2.0,0,0,3,7,201.599597,37.161216,1951,1991,98125,47.7210,-122.319,157.006138,709.686323
2,5631500400,2015-02-25,180000.0,2,1.00,71.535341,929.030400,1.0,0,0,3,6,71.535341,0.000000,1933,0,98028,47.7379,-122.233,252.696269,748.984308
3,2487200875,2014-12-09,604000.0,4,3.00,182.089958,464.515200,1.0,0,0,5,7,97.548192,84.541766,1965,0,98136,47.5208,-122.393,126.348134,464.515200
4,1954400510,2015-02-18,510000.0,3,2.00,156.077107,750.656563,1.0,0,0,3,8,156.077107,0.000000,1987,0,98074,47.6168,-122.045,167.225472,697.051509


Os dados são 21 613 vendas de casas no condado de King, no estado de Washington, que inclui Seattle, entre maio de 2014 e maio de 2015. Vêm do dataset [House Sales in King County, USA](https://www.kaggle.com/datasets/harlfoxem/housesalesprediction) no Kaggle, feito a partir dos registos públicos de vendas do condado.

As áreas vêm em pés quadrados e passamo-las para metros quadrados, com nomes novos. `AreaCasa` é a área da casa, `AreaTerreno` a do terreno, e `AreaAcimaChao` e `AreaCave` são a área acima do chão e a da cave. `AreaCasaVizinhos` e `AreaTerrenoVizinhos` são as mesmas áreas nas 15 casas vizinhas mais próximas. `grade` é a qualidade da construção numa escala de 1 a 13 e `condition` é o estado da casa de 1 a 5, as duas atribuídas pelo condado. `view` vai de 0 a 4 e diz quão boa é a vista, `waterfront` é 1 quando a casa fica à beira de água e `yr_renovated` é 0 quando a casa nunca foi renovada.

In [3]:
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nao_nulos": df.notna().sum(),
    "distintos": df.nunique(),
    "exemplo": df.iloc[0],
})
overview

,dtype,nao_nulos,distintos,exemplo
id,int64,21613,21436,7129300520
date,datetime64[us],21613,372,2014-10-13 00:00:00
price,float64,21613,4028,221900.0
bedrooms,int64,21613,13,3
bathrooms,float64,21613,30,1.0
AreaCasa,float64,21613,1038,109.625587
AreaTerreno,float64,21613,9782,524.902176
floors,float64,21613,6,1.0
waterfront,int64,21613,2,0
view,int64,21613,5,0


In [4]:
df.describe(percentiles=[.01, .25, .5, .75, .99]).T.round(2)

,count,mean,min,1%,25%,50%,75%,99%,max,std
id,21613.0,4580301520.864988,1000102.0,122033870.92,2123049194.0,3904930410.0,7308900445.0,9558728047.200001,9900000190.0,2876565571.312057
date,21613,2014-10-29 04:38:01.959931,2014-05-02 00:00:00,2014-05-06 00:00:00,2014-07-22 00:00:00,2014-10-16 00:00:00,2015-02-17 00:00:00,2015-05-07 00:00:00,2015-05-27 00:00:00,NaN
price,21613.0,540088.141767,75000.0,153500.36,321950.0,450000.0,645000.0,1964400.0,7700000.0,367127.196483
bedrooms,21613.0,3.370842,0.0,2.0,3.0,3.0,4.0,6.0,33.0,0.930062
bathrooms,21613.0,2.114757,0.0,1.0,1.75,2.25,2.5,4.25,8.0,0.770163
AreaCasa,21613.0,193.229008,26.941882,66.890189,132.572638,177.444806,236.902752,462.545656,1257.907162,85.325951
AreaTerreno,21613.0,1403.483212,48.309581,94.121928,468.231322,707.735359,992.947692,19789.090744,153416.271231,3848.091438
floors,21613.0,1.494309,1.0,1.0,1.0,1.5,2.0,3.0,3.5,0.539989
waterfront,21613.0,0.007542,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.086517
view,21613.0,0.234303,0.0,0.0,0.0,0.0,0.0,4.0,4.0,0.766318


In [5]:
# Preço mediano pela qualidade da construção, pelo estado, pela vista e por estar à beira de água
for col in ["grade", "condition", "view", "waterfront"]:
    display(df.groupby(col)["price"].agg(["count", "median"]).round(0).T)

grade,1,3,4,5,6,7,8,9,10,11,12,13
count,1.0,3.0,29.0,242.0,2038.0,8981.0,6068.0,2615.0,1134.0,399.0,90.0,13.0
median,142000.0,262000.0,205000.0,228700.0,275276.0,375000.0,510000.0,720000.0,914327.0,1284000.0,1817500.0,2983000.0


condition,1,2,3,4,5
count,30.0,172.0,14031.0,5679.0,1701.0
median,262500.0,279000.0,450000.0,440000.0,526000.0


view,0,1,2,3,4
count,19489.0,332.0,963.0,510.0,319.0
median,432500.0,690944.0,675000.0,802500.0,1185000.0


waterfront,0,1
count,21450.0,163.0
median,450000.0,1400000.0


In [6]:
# Preço por trimestre
df.groupby(df["date"].dt.to_period("Q"))["price"].agg(["count", "mean", "median"]).round(0)

,count,mean,median
date,,,
2014Q2,3948,553559.0,465000.0
2014Q3,5925,537406.0,450000.0
2014Q4,4760,529467.0,438975.0
2015Q1,4103,528655.0,440000.0
2015Q2,2877,561005.0,470500.0


In [7]:
# Os 5 códigos postais mais baratos e os 5 mais caros, pela mediana do preço
por_zip = df.groupby("zipcode")["price"].agg(["count", "median"]).sort_values("median")
print(f"{len(por_zip)} códigos postais")
pd.concat([por_zip.head(), por_zip.tail()])

70 códigos postais


,count,median
zipcode,,
98002,199,235000.0
98168,269,235000.0
98032,125,249000.0
98001,362,260000.0
98188,136,264000.0
98005,168,765475.0
98112,269,915000.0
98040,282,993750.0
98004,317,1150000.0


Não há valores em falta nem linhas repetidas. A área da casa é sempre a soma da área acima do chão com a da cave, por isso `AreaCasa`, `AreaAcimaChao` e `AreaCave` repetem informação.

In [8]:
print("Valores em falta:", df.isna().sum().sum())
print("Linhas exatamente iguais:", df.duplicated().sum())
print("Mesmo imóvel na mesma data:", df.duplicated(["id", "date"]).sum())
print("AreaCasa = AreaAcimaChao + AreaCave em todas as linhas:",
      np.isclose(df["AreaCasa"], df["AreaAcimaChao"] + df["AreaCave"]).all())

Valores em falta: 0
Linhas exatamente iguais: 0
Mesmo imóvel na mesma data: 0
AreaCasa = AreaAcimaChao + AreaCave em todas as linhas: True


Uma casa é identificada pelo `id`. 176 casas foram vendidas duas vezes no período, uma delas três vezes, e as características nunca mudam entre vendas. Entre a primeira e a última venda passaram em mediana 178 dias e o preço subiu em mediana 54%. Uma subida destas em seis meses, sem mudar `grade`, `condition` nem `yr_renovated`, sugere casas compradas para obras e revendidas, com as obras por registar.

In [9]:
repetidos = df[df.duplicated("id", keep=False)].sort_values(["id", "date"])
print("Imóveis vendidos mais do que uma vez:", repetidos["id"].nunique(), f"({len(repetidos)} linhas)")

# Algum imóvel muda de características entre vendas?
fisicas = ["bedrooms", "bathrooms", "AreaCasa", "AreaTerreno", "floors", "condition", "grade", "yr_renovated"]
print("Com características diferentes entre vendas:", (repetidos.groupby("id")[fisicas].nunique() > 1).any(axis=1).sum())

evolucao = repetidos.groupby("id").agg(primeira_data=("date", "first"), ultima_data=("date", "last"),
                                       primeiro_preco=("price", "first"), ultimo_preco=("price", "last"))
evolucao["dias"] = (evolucao["ultima_data"] - evolucao["primeira_data"]).dt.days
evolucao["variacao_pct"] = ((evolucao["ultimo_preco"] / evolucao["primeiro_preco"] - 1) * 100).round(1)
evolucao[["dias", "variacao_pct"]].describe().round(1)

Imóveis vendidos mais do que uma vez: 176 (353 linhas)
Com características diferentes entre vendas: 0


,dias,variacao_pct
count,176.0,176.0
mean,183.3,56.8
std,67.0,46.7
min,61.0,-5.4
25%,128.5,23.6
50%,177.5,54.4
75%,233.8,74.2
max,315.0,321.8


Procuramos valores impossíveis ou pouco prováveis numa casa.

In [10]:
checks = {
    "bedrooms == 0": df["bedrooms"] == 0,
    "bathrooms == 0": df["bathrooms"] == 0,
    "bedrooms >= 10": df["bedrooms"] >= 10,
    "AreaCasa < 50 m²": df["AreaCasa"] < 50,
    "AreaTerreno > 4 hectares": df["AreaTerreno"] > 40_000,
    "grade <= 4": df["grade"] <= 4,
    "yr_built > ano da venda": df["yr_built"] > df["date"].dt.year,
    "price < 100k": df["price"] < 100_000,
}
pd.Series({k: v.sum() for k, v in checks.items()}, name="linhas").to_frame()

,linhas
bedrooms == 0,13
bathrooms == 0,10
bedrooms >= 10,5
AreaCasa < 50 m²,27
AreaTerreno > 4 hectares,35
grade <= 4,33
yr_built > ano da venda,12
price < 100k,25


In [11]:
colunas = ["price", "bedrooms", "bathrooms", "AreaCasa", "AreaTerreno", "floors", "grade", "yr_built"]
df.loc[checks["bedrooms >= 10"] | checks["bedrooms == 0"] | checks["bathrooms == 0"], colunas].sort_values("bedrooms")

,price,bedrooms,bathrooms,AreaCasa,AreaTerreno,floors,grade,yr_built
875,1095000.0,0,0.00,284.654915,442.590083,3.5,7,1990
18379,265000.0,0,0.75,35.674767,19829.596470,1.0,4,2003
14423,139950.0,0,0.00,78.410166,396.603078,1.0,7,1913
12653,320000.0,0,2.50,138.425530,660.633517,2.0,7,1999
9854,235000.0,0,0.00,136.567469,445.934592,2.0,7,1996
8484,240000.0,0,2.50,168.154502,526.667334,2.0,7,2003
8477,339950.0,0,2.50,212.747962,772.860390,2.0,8,1985
9773,355000.0,0,0.00,228.541478,747.776569,2.0,8,1990
4868,228000.0,0,1.00,36.232186,548.127936,1.0,4,1953
3467,288000.0,0,1.50,132.851347,153.290016,3.0,7,1999


In [12]:
df.loc[checks["AreaCasa < 50 m²"] | checks["grade <= 4"] | checks["price < 100k"], colunas + ["condition"]].sort_values("AreaCasa")

,price,bedrooms,bathrooms,AreaCasa,AreaTerreno,floors,grade,yr_built,condition
19452,142000.0,0,0.00,26.941882,1939.350960,1.0,1,1963,1
15381,276000.0,1,0.75,34.374125,167.318375,1.0,5,1923,5
860,245000.0,1,0.75,35.303155,1393.545600,1.0,5,1963,3
18379,265000.0,0,0.75,35.674767,19829.596470,1.0,4,2003,3
21332,245000.0,1,1.00,36.232186,185.806080,1.0,6,1920,4
...,...,...,...,...,...,...,...,...,...
14581,95000.0,2,1.00,89.186918,650.321280,1.0,4,1918,3
18939,90000.0,3,1.00,91.044979,231.328570,2.0,6,1969,4
7973,120000.0,2,1.00,91.974010,3712.777091,1.0,4,1945,2
10585,95000.0,2,1.00,99.406253,1899.867168,1.0,6,1948,2


O único erro claro é a casa com 33 quartos em 150 m² e 1,75 casas de banho, que provavelmente tem 3. As casas com 0 quartos ou 0 casas de banho são poucas e têm áreas normais, por isso devem ser valores por registar. As casas muito pequenas, com `grade` baixo ou vendidas abaixo de 100 mil dólares são sobretudo casas antigas e de construção simples, de 1945 em mediana, o que é plausível. Os terrenos acima de 4 hectares ficam em zonas rurais do condado como Enumclaw, Vashon e Carnation. As 12 casas construídas depois da data da venda podem ter sido vendidas antes de estarem acabadas ou ter o ano mal registado. Fora os 33 quartos, mantemos estes valores.

## Preparação dos dados

Corrigimos só o que é um erro claro, os 33 quartos, que passam a 3.

In [13]:
dados = df.copy()

# 33 quartos numa casa de 150 m² com 1,75 casas de banho deve ser um erro de escrita de 3
erro_quartos = dados["bedrooms"] == 33
dados.loc[erro_quartos, "bedrooms"] = 3
print(f"bedrooms corrigidos: {erro_quartos.sum()} | linhas finais: {len(dados)}")

bedrooms corrigidos: 1 | linhas finais: 21613


Para saber se uma casa foi vendida acima ou abaixo do normal, estimamos o preço esperado com uma regressão Ridge sobre o log do preço e comparamos com o preço real.

A regressão usa `AreaCasa`, `AreaTerreno`, `bedrooms`, `bathrooms`, o código postal e o mês da venda. As áreas entram como curvas e o resto como categorias. Não usamos a qualidade, o estado, a vista, o ano de construção nem as renovações, porque são as variáveis com que queremos explicar o desvio ao preço esperado. O preço esperado de cada venda vem de validação cruzada com 5 partes, de um modelo que não viu essa venda.

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, SplineTransformer

# Áreas em log, porque o preço não cresce em linha reta com a área
dados["LogAreaCasa"] = np.log(dados["AreaCasa"])
dados["LogAreaTerreno"] = np.log(dados["AreaTerreno"])

# Quartos e casas de banho como categorias, com 6 ou mais quartos e 5 ou mais casas de banho juntos
dados["QuartosCat"] = dados["bedrooms"].clip(upper=6).astype(str)
dados["BanhosCat"] = dados["bathrooms"].clip(upper=5).round().astype(int).astype(str)
dados["CodigoPostal"] = dados["zipcode"].astype(str)
dados["MesVenda"] = dados["date"].dt.to_period("M").astype(str)

spline_cols = ["LogAreaCasa", "LogAreaTerreno"]
cat_cols = ["QuartosCat", "BanhosCat", "CodigoPostal", "MesVenda"]
colunas_auxiliares = spline_cols + cat_cols

modelo = make_pipeline(
    ColumnTransformer([
        ("spline", SplineTransformer(n_knots=6, degree=3), spline_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]),
    Ridge(alpha=1.0),
)

X, y = dados[spline_cols + cat_cols], np.log(dados["price"])
y_prev = cross_val_predict(modelo, X, y, cv=KFold(5, shuffle=True, random_state=42))

r2 = 1 - ((y - y_prev) ** 2).sum() / ((y - y.mean()) ** 2).sum()
erro_pct = (np.exp(np.abs(y - y_prev)) - 1) * 100
print(f"R² (log do preço, out-of-fold): {r2:.3f}")
print(f"Erro mediano: {np.median(erro_pct):.1f}% do preço")

R² (log do preço, out-of-fold): 0.841
Erro mediano: 11.8% do preço


O desvio de cada venda é a diferença, em %, entre o preço real e o esperado. O limite é 12%, perto do erro mediano da regressão, para que uma venda só seja `alto` ou `baixo` quando se afasta do esperado mais do que a regressão costuma errar. Abaixo de −12% a venda é `baixo`, acima de +12% é `alto` e entre os dois é `normal`.

In [15]:
dados["PrecoEsperado"] = np.exp(y_prev).round(0)
dados["DesvioPreco"] = (dados["price"] / dados["PrecoEsperado"] - 1) * 100  # em %

LIMITE = 12  # %
dados["ClassePreco"] = pd.cut(dados["DesvioPreco"], [-np.inf, -LIMITE, LIMITE, np.inf],
                              labels=["baixo", "normal", "alto"])

print(f"baixo: desvio < -{LIMITE}% | normal: entre -{LIMITE}% e +{LIMITE}% | alto: desvio > +{LIMITE}%")
contagem = dados["ClassePreco"].value_counts().to_frame()
contagem["pct_vendas"] = (dados["ClassePreco"].value_counts(normalize=True) * 100).round(1)
contagem

baixo: desvio < -12% | normal: entre -12% e +12% | alto: desvio > +12%


,count,pct_vendas
ClassePreco,,
normal,11430,52.9
alto,5335,24.7
baixo,4848,22.4


In [16]:
# Desvio por classe
dados.groupby("ClassePreco", observed=True)["DesvioPreco"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
ClassePreco,,,,,,,,
baixo,4848.0,-23.0,9.6,-72.6,-27.7,-20.6,-15.6,-12.0
normal,11430.0,0.1,6.6,-12.0,-5.4,0.1,5.6,12.0
alto,5335.0,29.9,23.9,12.0,16.3,22.3,33.6,251.4


In [17]:
# Percentagem de cada classe pela qualidade da construção, pelo estado e pela vista
for col in ["grade", "condition", "view"]:
    display((pd.crosstab(dados[col], dados["ClassePreco"], normalize="index") * 100).round(1))

ClassePreco,baixo,normal,alto
grade,,,
1,100.0,0.0,0.0
3,33.3,0.0,66.7
4,51.7,10.3,37.9
5,45.0,31.8,23.1
6,35.5,42.3,22.2
7,25.5,55.7,18.8
8,18.5,56.2,25.3
9,11.9,52.9,35.2
10,17.5,44.5,38.0


ClassePreco,baixo,normal,alto
condition,,,
1,66.7,13.3,20.0
2,52.3,27.9,19.8
3,22.5,54.1,23.4
4,23.3,51.8,25.0
5,15.6,49.6,34.7


ClassePreco,baixo,normal,alto
view,,,
0,23.4,54.9,21.6
1,18.4,38.9,42.8
2,15.5,39.4,45.2
3,11.8,32.9,55.3
4,4.1,14.4,81.5


Cerca de metade das vendas são `normal` e o resto divide-se quase por igual entre `baixo` e `alto`. A qualidade da construção, o estado e a vista mudam bastante a proporção das classes. 82% das casas com a melhor vista e 59% das casas com `grade` 12 são `alto`, e mais de metade das casas em estado 1 ou 2 são `baixo`.

In [18]:
# Exemplos: as vendas mais acima e mais abaixo do esperado
cols = ["zipcode", "bedrooms", "bathrooms", "AreaCasa", "grade", "condition", "price", "PrecoEsperado", "DesvioPreco", "ClassePreco"]
display(dados.nlargest(5, "DesvioPreco")[cols].round(1))
dados.nsmallest(5, "DesvioPreco")[cols].round(1)

,zipcode,bedrooms,bathrooms,AreaCasa,grade,condition,price,PrecoEsperado,DesvioPreco,ClassePreco
1081,98178,2,1.0,106.8,6,4,825000.0,234806.0,251.4,alto
20325,98056,3,3.5,409.7,11,3,3000000.0,872735.0,243.7,alto
6402,98118,5,2.5,348.4,10,3,2903000.0,859746.0,237.7,alto
16252,98034,3,2.5,233.2,10,3,1950000.0,591003.0,229.9,alto
4013,98075,3,2.2,278.7,11,3,2400000.0,737722.0,225.3,alto


,zipcode,bedrooms,bathrooms,AreaCasa,grade,condition,price,PrecoEsperado,DesvioPreco,ClassePreco
16893,98117,3,1.0,114.3,6,1,125000.0,456632.0,-72.6,baixo
18332,98116,3,1.0,111.5,7,1,130000.0,447388.0,-70.9,baixo
1222,98033,3,1.0,103.1,7,4,130000.0,446093.0,-70.9,baixo
21372,98119,5,3.5,414.3,10,3,490000.0,1611089.0,-69.6,baixo
14835,98119,3,2.5,220.2,9,3,300523.0,945651.0,-68.2,baixo


Criamos sete variáveis a partir das colunas que já existem. `Renovada` é 1 quando a casa foi renovada e `AnoUltimaObra` é o ano da renovação, ou o da construção se a casa nunca foi renovada. `FracaoCave` é a parte da área da casa que é cave, porque a regressão conta a cave como área normal. `AreaVsVizinhos` e `LoteVsVizinhos` comparam a área da casa e do terreno com as das 15 casas vizinhas mais próximas. `AnosSemObras` são os anos entre a última obra, a renovação ou a construção, e a venda. `AreaPorQuarto` é a área da casa a dividir pelo número de quartos, contando pelo menos um quarto.

In [19]:
# Ano da última obra: o da renovação, ou o da construção se a casa nunca foi renovada
dados["Renovada"] = (dados["yr_renovated"] > 0).astype(int)
dados["AnoUltimaObra"] = dados["yr_renovated"].where(dados["yr_renovated"] > 0, dados["yr_built"])

dados["FracaoCave"] = dados["AreaCave"] / dados["AreaCasa"]
dados["AreaVsVizinhos"] = dados["AreaCasa"] / dados["AreaCasaVizinhos"]
dados["LoteVsVizinhos"] = dados["AreaTerreno"] / dados["AreaTerrenoVizinhos"]
dados["AnosSemObras"] = dados["date"].dt.year - dados["AnoUltimaObra"]
dados["AreaPorQuarto"] = dados["AreaCasa"] / dados["bedrooms"].clip(lower=1)

NOVAS = ["Renovada", "AnoUltimaObra", "FracaoCave", "AreaVsVizinhos", "LoteVsVizinhos", "AnosSemObras", "AreaPorQuarto"]
dados[NOVAS].describe(percentiles=[.01, .5, .99]).T.round(2)

,count,mean,std,min,1%,50%,99%,max
Renovada,21613.0,0.04,0.20,0.00,0.00,0.00,1.00,1.00
AnoUltimaObra,21613.0,1973.39,28.81,1900.00,1905.00,1977.00,2014.00,2015.00
FracaoCave,21613.0,0.12,0.17,0.00,0.00,0.00,0.50,0.67
AreaVsVizinhos,21613.0,1.05,0.32,0.18,0.47,1.00,2.18,6.00
LoteVsVizinhos,21613.0,1.13,1.28,0.05,0.39,1.00,3.72,87.53
AnosSemObras,21613.0,40.94,28.81,-1.00,0.00,37.00,110.00,115.00
AreaPorQuarto,21613.0,57.49,20.39,15.17,27.87,53.57,123.30,446.86


## Guardar o dataset preparado

A pasta `data/` está no `.gitignore`, por isso o ficheiro é gerado ao correr este notebook.

In [20]:
saida = "data/kingcounty/kc_clean.csv"
dados.drop(columns=colunas_auxiliares).to_csv(saida, index=False)
print("Guardado em", saida, dados.shape)

Guardado em data/kingcounty/kc_clean.csv (21613, 37)
